In [8]:

import pandas as pd

df = pd.read_csv("/content/train.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column names:", df.columns.tolist())

df.head()


Rows: 891
Columns: 12
Column names: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:

import pandas as pd

def audit_csv(df):
    report = {
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "empty_rows": int(df.isna().all(axis=1).sum()),
        "empty_columns": int(df.isna().all(axis=0).sum()),
        "missing_values": df.isna().sum().to_dict(),
        "column_types": df.dtypes.astype(str).to_dict(),
        "invalid_emails": 0,
        "invalid_ages": 0,
    }

    if "Email" in df.columns:
        emails = df["Email"].dropna().astype(str)
        valid = emails.str.fullmatch(
            r"[^@\s]+@[^@\s]+\.[^@\s]+"
        )
        report["invalid_emails"] = int((~valid).sum())

    if "Age" in df.columns:
        ages = pd.to_numeric(df["Age"], errors="coerce")
        invalid = ages.notna() & ~ages.between(0, 120)
        report["invalid_ages"] = int(invalid.sum())

    return report


def display_report(report):
    print("===== CLEANSHEET AUDIT REPORT =====")
    print("Rows:", report["total_rows"])
    print("Columns:", report["total_columns"])
    print("Duplicate rows:", report["duplicate_rows"])
    print("Empty rows:", report["empty_rows"])
    print("Empty columns:", report["empty_columns"])
    print("Missing values:", report["missing_values"])
    print("Column types:", report["column_types"])
    print("Invalid emails:", report["invalid_emails"])
    print("Invalid ages:", report["invalid_ages"])


report = audit_csv(df)
display_report(report)


===== CLEANSHEET AUDIT REPORT =====
Rows: 891
Columns: 12
Duplicate rows: 0
Empty rows: 0
Empty columns: 0
Missing values: {'PassengerId': 0, 'Survived': 0, 'Pclass': 0, 'Name': 0, 'Sex': 0, 'Age': 177, 'SibSp': 0, 'Parch': 0, 'Ticket': 0, 'Fare': 0, 'Cabin': 687, 'Embarked': 2}
Column types: {'PassengerId': 'int64', 'Survived': 'int64', 'Pclass': 'int64', 'Name': 'object', 'Sex': 'object', 'Age': 'float64', 'SibSp': 'int64', 'Parch': 'int64', 'Ticket': 'object', 'Fare': 'float64', 'Cabin': 'object', 'Embarked': 'object'}
Invalid emails: 0
Invalid ages: 0


In [10]:

def missing_value_report(df):
    missing_count = df.isna().sum()
    missing_percentage = (
        missing_count / len(df) * 100
    ).round(2)

    summary = pd.DataFrame({
        "Missing Values": missing_count,
        "Missing Percentage": missing_percentage
    })

    summary = summary[
        summary["Missing Values"] > 0
    ]

    return summary.sort_values(
        by="Missing Percentage",
        ascending=False
    )


missing_report = missing_value_report(df)
display(missing_report)


,Missing Values,Missing Percentage
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


In [11]:

def generate_quality_warnings(df):
    warnings = []

    # Check for missing values
    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        if missing_count > 0:
            missing_percentage = (
                missing_count / len(df) * 100
            )

            if missing_percentage >= 50:
                severity = "HIGH"
            elif missing_percentage >= 10:
                severity = "MEDIUM"
            else:
                severity = "LOW"

            warnings.append({
                "Column": column,
                "Issue": "Missing values",
                "Count": missing_count,
                "Percentage": round(missing_percentage, 2),
                "Severity": severity
            })

    # Check for duplicate rows
    duplicate_count = int(df.duplicated().sum())

    if duplicate_count > 0:
        warnings.append({
            "Column": "Entire dataset",
            "Issue": "Duplicate rows",
            "Count": duplicate_count,
            "Percentage": round(
                duplicate_count / len(df) * 100, 2
            ),
            "Severity": "MEDIUM"
        })

    return pd.DataFrame(warnings)


warnings_df = generate_quality_warnings(df)

if warnings_df.empty:
    print("No data-quality issues detected.")
else:
    display(warnings_df)


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,177,19.87,MEDIUM
1,Cabin,Missing values,687,77.10,HIGH
2,Embarked,Missing values,2,0.22,LOW


In [12]:

# Export the quality warnings to a CSV file

output_file = "/content/cleansheet_quality_report.csv"

warnings_df.to_csv(output_file, index=False)

print("Report exported successfully!")
print("File location:", output_file)

# Preview the exported report
exported_report = pd.read_csv(output_file)
display(exported_report)


Report exported successfully!
File location: /content/cleansheet_quality_report.csv


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,177,19.87,MEDIUM
1,Cabin,Missing values,687,77.10,HIGH
2,Embarked,Missing values,2,0.22,LOW


In [13]:

# Create a small dataset with intentional quality issues

test_df = pd.DataFrame({
    "Name": ["Anu", "Ravi", "Anu", "Maya", "John"],
    "Age": [20, -5, 20, None, 150],
    "Email": [
        "anu@example.com",
        "ravi.example.com",
        "anu@example.com",
        None,
        "john@example.com"
    ]
})

print("TEST DATASET:")
display(test_df)

print("\nBASIC AUDIT:")
test_report = audit_csv(test_df)
display_report(test_report)

print("\nQUALITY WARNINGS:")
test_warnings = generate_quality_warnings(test_df)
display(test_warnings)


TEST DATASET:


,Name,Age,Email
0,Anu,20.0,anu@example.com
1,Ravi,-5.0,ravi.example.com
2,Anu,20.0,anu@example.com
3,Maya,NaN,None
4,John,150.0,john@example.com



BASIC AUDIT:
===== CLEANSHEET AUDIT REPORT =====
Rows: 5
Columns: 3
Duplicate rows: 1
Empty rows: 0
Empty columns: 0
Missing values: {'Name': 0, 'Age': 1, 'Email': 1}
Column types: {'Name': 'object', 'Age': 'float64', 'Email': 'object'}
Invalid emails: 1
Invalid ages: 2

QUALITY WARNINGS:


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,1,20.0,MEDIUM
1,Email,Missing values,1,20.0,MEDIUM
2,Entire dataset,Duplicate rows,1,20.0,MEDIUM


In [14]:

def generate_quality_warnings(df):
    warnings = []

    def add_warning(column, issue, count, percentage, severity):
        warnings.append({
            "Column": column,
            "Issue": issue,
            "Count": count,
            "Percentage": round(percentage, 2),
            "Severity": severity
        })

    def get_severity(percentage):
        if percentage >= 50:
            return "HIGH"
        elif percentage >= 10:
            return "MEDIUM"
        return "LOW"

    # 1. Missing values
    for column in df.columns:
        count = int(df[column].isna().sum())

        if count > 0:
            percentage = count / len(df) * 100
            add_warning(
                column, "Missing values", count,
                percentage, get_severity(percentage)
            )

    # 2. Duplicate rows
    duplicates = int(df.duplicated().sum())

    if duplicates > 0:
        percentage = duplicates / len(df) * 100
        add_warning(
            "Entire dataset", "Duplicate rows", duplicates,
            percentage, get_severity(percentage)
        )

    # 3. Invalid ages
    if "Age" in df.columns:
        ages = pd.to_numeric(df["Age"], errors="coerce")
        invalid = ages.notna() & ~ages.between(0, 120)
        count = int(invalid.sum())

        if count > 0:
            percentage = count / len(df) * 100
            add_warning(
                "Age", "Invalid ages", count,
                percentage, get_severity(percentage)
            )

    # 4. Invalid email addresses
    if "Email" in df.columns:
        emails = df["Email"].dropna().astype(str)
        valid = emails.str.fullmatch(
            r"[^@\s]+@[^@\s]+\.[^@\s]+"
        )
        count = int((~valid).sum())

        if count > 0:
            percentage = count / len(df) * 100
            add_warning(
                "Email", "Invalid emails", count,
                percentage, get_severity(percentage)
            )

    return pd.DataFrame(
        warnings,
        columns=[
            "Column", "Issue", "Count",
            "Percentage", "Severity"
        ]
    )


In [15]:

test_warnings = generate_quality_warnings(test_df)
display(test_warnings)


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,1,20.0,MEDIUM
1,Email,Missing values,1,20.0,MEDIUM
2,Entire dataset,Duplicate rows,1,20.0,MEDIUM
3,Age,Invalid ages,2,40.0,MEDIUM
4,Email,Invalid emails,1,20.0,MEDIUM


In [16]:

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


Dataset shape: (891, 12)

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicate rows: 0


In [17]:

print("Synthetic test shape:", test_df.shape)
print("Duplicate rows:", test_df.duplicated().sum())

ages = pd.to_numeric(test_df["Age"], errors="coerce")
print("Invalid ages:", int((ages.notna() & ~ages.between(0, 120)).sum()))

emails = test_df["Email"].dropna().astype(str)
valid_emails = emails.str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+")
print("Invalid emails:", int((~valid_emails).sum()))


Synthetic test shape: (5, 3)
Duplicate rows: 1
Invalid ages: 2
Invalid emails: 1


In [19]:
from google.colab import files
files.download("/content/cleansheet_quality_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from google.colab import files
files.download("/content/cleansheet_quality_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import pandas as pd

report = pd.read_csv("/content/cleansheet_quality_report.csv")

print("Number of columns:", len(report.columns))
print("Column names:", report.columns.tolist())

display(report.head())

Number of columns: 5
Column names: ['Column', 'Issue', 'Count', 'Percentage', 'Severity']


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,177,19.87,MEDIUM
1,Cabin,Missing values,687,77.10,HIGH
2,Embarked,Missing values,2,0.22,LOW


In [24]:

import pandas as pd

def clean_dataset(
    df,
    remove_duplicates=True,
    fill_missing_numeric=True,
    fill_missing_categorical=False,
    validate_ages=True,
    handle_invalid_emails=True
):
    # Work on a copy so the original data stays unchanged.
    cleaned_df = df.copy(deep=True)
    cleaning_log = []

    # 1. Remove duplicate rows.
    if remove_duplicates:
        before = len(cleaned_df)
        cleaned_df = cleaned_df.drop_duplicates()
        removed = before - len(cleaned_df)

        cleaning_log.append({
            "Action": "Remove duplicate rows",
            "Affected": removed
        })

    # 2. Treat impossible ages as missing values.
    if validate_ages:
        age_columns = [
            col for col in cleaned_df.columns
            if col.strip().lower() == "age"
        ]

        for col in age_columns:
            cleaned_df[col] = pd.to_numeric(
                cleaned_df[col], errors="coerce"
            )

            invalid = (
                cleaned_df[col].notna()
                & ~cleaned_df[col].between(0, 120)
            )
            count = int(invalid.sum())
            cleaned_df.loc[invalid, col] = pd.NA

            cleaning_log.append({
                "Action": f"Replace invalid values in {col} with missing",
                "Affected": count
            })

    # 3. Replace invalid email formats with missing values.
    if handle_invalid_emails:
        email_columns = [
            col for col in cleaned_df.columns
            if "email" in col.strip().lower()
        ]

        for col in email_columns:
            values = cleaned_df[col].astype("string").str.strip()
            present = cleaned_df[col].notna() & values.ne("")

            valid = values.str.fullmatch(
                r"[^@\s]+@[^@\s]+\.[^@\s]+",
                na=False
            )
            invalid = present & ~valid
            count = int(invalid.sum())

            cleaned_df.loc[invalid, col] = pd.NA

            cleaning_log.append({
                "Action": f"Replace invalid emails in {col} with missing",
                "Affected": count
            })

    # 4. Fill missing numeric values with the column median.
    if fill_missing_numeric:
        numeric_columns = cleaned_df.select_dtypes(
            include="number"
        ).columns

        for col in numeric_columns:
            missing = int(cleaned_df[col].isna().sum())

            if missing > 0 and cleaned_df[col].notna().any():
                median_value = cleaned_df[col].median()
                cleaned_df[col] = cleaned_df[col].fillna(median_value)

                cleaning_log.append({
                    "Action": f"Fill missing {col} with median",
                    "Affected": missing
                })

    # 5. Optionally fill missing categorical values with the mode.
    if fill_missing_categorical:
        categorical_columns = cleaned_df.select_dtypes(
            include=["object", "category", "string"]
        ).columns

        for col in categorical_columns:
            missing = int(cleaned_df[col].isna().sum())
            non_missing = cleaned_df[col].dropna()

            if missing > 0 and not non_missing.empty:
                mode_values = non_missing.mode()

                if not mode_values.empty:
                    cleaned_df[col] = cleaned_df[col].fillna(
                        mode_values.iloc[0]
                    )

                    cleaning_log.append({
                        "Action": f"Fill missing {col} with mode",
                        "Affected": missing
                    })

    return cleaned_df, pd.DataFrame(
        cleaning_log,
        columns=["Action", "Affected"]
    )


In [25]:

test_df = pd.DataFrame({
    "Name": ["Anu", "Ravi", "Anu", "Maya", "John"],
    "Age": [20, -5, 20, None, 150],
    "Email": [
        "anu@example.com",
        "ravi.example.com",
        "anu@example.com",
        None,
        "john@example.com"
    ]
})

cleaned_test_df, cleaning_log = clean_dataset(test_df)

print("ORIGINAL DATA")
display(test_df)

print("CLEANED DATA")
display(cleaned_test_df)

print("CLEANING LOG")
display(cleaning_log)


ORIGINAL DATA


,Name,Age,Email
0,Anu,20.0,anu@example.com
1,Ravi,-5.0,ravi.example.com
2,Anu,20.0,anu@example.com
3,Maya,NaN,None
4,John,150.0,john@example.com


CLEANED DATA


,Name,Age,Email
0,Anu,20.0,anu@example.com
1,Ravi,20.0,<NA>
3,Maya,20.0,None
4,John,20.0,john@example.com


CLEANING LOG


,Action,Affected
0,Remove duplicate rows,1
1,Replace invalid values in Age with missing,2
2,Replace invalid emails in Email with missing,1
3,Fill missing Age with median,3


In [26]:

# Test 1: Confirm the original data was not modified
assert len(test_df) == 5
assert test_df.loc[1, "Age"] == -5
assert test_df.loc[2, "Name"] == "Anu"

print("PASS: Original dataset remains unchanged.")

# Test 2: Keep duplicates and do not fill missing numeric values
custom_cleaned, custom_log = clean_dataset(
    test_df,
    remove_duplicates=False,
    fill_missing_numeric=False
)

assert len(custom_cleaned) == 5
assert custom_cleaned["Age"].isna().sum() == 3

print("PASS: Custom cleaning options work.")
display(custom_cleaned)


PASS: Original dataset remains unchanged.
PASS: Custom cleaning options work.


,Name,Age,Email
0,Anu,20.0,anu@example.com
1,Ravi,NaN,<NA>
2,Anu,20.0,anu@example.com
3,Maya,NaN,None
4,John,NaN,john@example.com


In [27]:

# STEP 4: Run the complete CleanSheet workflow

# 1. Preserve the original dataset
original_df = df.copy(deep=True)

# 2. Generate the quality report BEFORE cleaning
warnings_df = generate_quality_warnings(original_df)

# 3. Clean a copy of the dataset
cleaned_df, cleaning_log = clean_dataset(
    original_df,
    remove_duplicates=True,
    fill_missing_numeric=True,
    fill_missing_categorical=False,
    validate_ages=True,
    handle_invalid_emails=True
)

# 4. Save the cleaned dataset
cleaned_file = "/content/cleansheet_cleaned.csv"
cleaned_df.to_csv(cleaned_file, index=False)

# 5. Save the original quality report
report_file = "/content/cleansheet_quality_report.csv"
warnings_df.to_csv(report_file, index=False)

# 6. Save a log of the cleaning actions
log_file = "/content/cleansheet_cleaning_log.csv"
cleaning_log.to_csv(log_file, index=False)

# 7. Display results
print("CleanSheet completed successfully!")
print("Original rows:", len(original_df))
print("Cleaned rows:", len(cleaned_df))
print("Rows removed:", len(original_df) - len(cleaned_df))

print("\nQUALITY REPORT")
display(warnings_df)

print("\nCLEANING LOG")
display(cleaning_log)

print("\nFiles saved:")
print(cleaned_file)
print(report_file)
print(log_file)


CleanSheet completed successfully!
Original rows: 891
Cleaned rows: 891
Rows removed: 0

QUALITY REPORT


,Column,Issue,Count,Percentage,Severity
0,Age,Missing values,177,19.87,MEDIUM
1,Cabin,Missing values,687,77.10,HIGH
2,Embarked,Missing values,2,0.22,LOW



CLEANING LOG


,Action,Affected
0,Remove duplicate rows,0
1,Replace invalid values in Age with missing,0
2,Fill missing Age with median,177



Files saved:
/content/cleansheet_cleaned.csv
/content/cleansheet_quality_report.csv
/content/cleansheet_cleaning_log.csv


In [28]:

import zipfile
from google.colab import files

zip_path = "/content/cleansheet_outputs.zip"

with zipfile.ZipFile(zip_path, "w") as zip_file:
    zip_file.write(
        "/content/cleansheet_cleaned.csv",
        arcname="cleansheet_cleaned.csv"
    )
    zip_file.write(
        "/content/cleansheet_quality_report.csv",
        arcname="cleansheet_quality_report.csv"
    )
    zip_file.write(
        "/content/cleansheet_cleaning_log.csv",
        arcname="cleansheet_cleaning_log.csv"
    )

print("Your CleanSheet output files are ready!")
files.download(zip_path)


Your CleanSheet output files are ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
import zipfile

# Customization options
remove_duplicates_box = widgets.Checkbox(
    value=True,
    description="Remove duplicate rows"
)

fill_numeric_box = widgets.Checkbox(
    value=True,
    description="Fill missing numeric values with median"
)

fill_categorical_box = widgets.Checkbox(
    value=False,
    description="Fill missing text values with mode"
)

validate_ages_box = widgets.Checkbox(
    value=True,
    description="Validate ages (0–120)"
)

validate_emails_box = widgets.Checkbox(
    value=True,
    description="Validate email formats"
)

run_button = widgets.Button(
    description="Clean My Dataset",
    button_style="success",
    icon="check"
)

output = widgets.Output()

def run_cleansheet(button):
    with output:
        clear_output()

        # Always work from the original dataset
        source_df = original_df.copy(deep=True)

        # Create a fresh quality report BEFORE cleaning
        current_report = generate_quality_warnings(source_df)

        # Apply the user's selected options
        result_df, log_df = clean_dataset(
            source_df,
            remove_duplicates=remove_duplicates_box.value,
            fill_missing_numeric=fill_numeric_box.value,
            fill_missing_categorical=fill_categorical_box.value,
            validate_ages=validate_ages_box.value,
            handle_invalid_emails=validate_emails_box.value
        )

        # Export all three outputs
        result_df.to_csv(
            "/content/cleansheet_cleaned.csv", index=False
        )
        current_report.to_csv(
            "/content/cleansheet_quality_report.csv", index=False
        )
        log_df.to_csv(
            "/content/cleansheet_cleaning_log.csv", index=False
        )

        print("CleanSheet completed!")
        print("Original rows:", len(source_df))
        print("Cleaned rows:", len(result_df))
        print("Rows removed:", len(source_df) - len(result_df))

        print("\nQuality report (before cleaning)")
        display(current_report)

        print("\nCleaning log")
        display(log_df)

        print("\nOutput files are ready for download.")

run_button.on_click(run_cleansheet)

display(widgets.VBox([
    widgets.HTML("<h3>CleanSheet: Cleaning Settings</h3>"),
    remove_duplicates_box,
    fill_numeric_box,
    fill_categorical_box,
    validate_ages_box,
    validate_emails_box,
    run_button,
    output
]))


,Action,Affected
0,Remove duplicate rows,0
1,Replace invalid values in Age with missing,0
2,Fill missing Age with median,177



Output files are ready for download.


In [30]:

import os
import zipfile
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

download_button = widgets.Button(
    description="Download All CleanSheet Outputs",
    button_style="primary",
    icon="download"
)

download_status = widgets.Output()

def download_outputs(button):
    with download_status:
        download_status.clear_output()

        required_files = [
            "/content/cleansheet_cleaned.csv",
            "/content/cleansheet_quality_report.csv",
            "/content/cleansheet_cleaning_log.csv"
        ]

        missing_files = [
            path for path in required_files
            if not os.path.exists(path)
        ]

        if missing_files:
            print("Please click 'Clean My Dataset' first.")
            return

        zip_path = "/content/cleansheet_outputs.zip"

        with zipfile.ZipFile(
            zip_path, "w", zipfile.ZIP_DEFLATED
        ) as zip_file:
            for path in required_files:
                zip_file.write(path, arcname=os.path.basename(path))

        print("Preparing your download...")
        files.download(zip_path)

download_button.on_click(download_outputs)

display(download_button, download_status)


Preparing your download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>